## Решение заданий с помощью Spark.SQL

Установим Spark подключение и проверим его.

In [11]:
# import pyspark as spark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Hadoop_Assignment_VSCode") \
    .enableHiveSupport() \
    .getOrCreate()


In [12]:
spark.sql("USE s_yudin")

DataFrame[]

In [13]:
#test
spark.sql("""
    SELECT * FROM dim_cat LIMIT 5
""").show()

+------+--------------------+
|cat_id|       category_name|
+------+--------------------+
|     1|                  $1|
|     2|                 $10|
|     3|          $10, Eagle|
|     4|                $100|
|     5|$2.50, Quarter Eagle|
+------+--------------------+



### 1

Рассчитать количество товаров, среднюю цену и максимальную цену для каждой category_name

---

In [6]:
spark.sql("""
    SELECT 
        category_name,
        COUNT(*) AS item_count,
        AVG(price) AS avg_price,
        MAX(price) AS max_price
    FROM ebay_listings_optimized
    WHERE price > 0
    GROUP BY category_name
    ORDER BY category_name
""").show()

+--------------------+----------+------------------+---------+
|       category_name|item_count|         avg_price|max_price|
+--------------------+----------+------------------+---------+
|                NULL|       185|50.944810810810786|   238.06|
|                  $1|        62|147.39661290322584|   990.76|
|                 $10|         8|          280.8275|   977.98|
|          $10, Eagle|        23| 31.97826086956522|    34.99|
|                $100|         7|115.11142857142859|   604.72|
|$2.50, Quarter Eagle|         2|             57.97|    58.28|
|   1 Pfennig ab 1950|         1|             23.12|    23.12|
|10-Euro-Gedenkmünzen|         1|             21.79|    21.79|
|                1000|       101| 61.96999999999999|    158.8|
|              100005|         4|           30.0925|    54.38|
|              100011|         1|             71.99|    71.99|
|              100012|         2|             32.87|    37.93|
|              100058|        38| 90.42894736842106|   

### Пояснение

- **Что считается:** для каждой `category_name` подсчитывается число товаров (`COUNT(*)`), средняя цена (`AVG(price)`) и максимальная цена (`MAX(price)`).  
- **Почему формула корректна:** `COUNT(*)` игнорирует NULL? Нет, считает все строки. Но `price > 0` отсекает невалидные цены. Среднее и максимум корректны при наличии хотя бы одной цены.  
- **Проверка:** Выполнить выборочно для одной категории:  
  ```sql
  SELECT price FROM ebay_listings_optimized WHERE category_name = 'Electronics' AND price > 0;
  ```  
  Сравнить вручную количество, среднее и максимум с результатом агрегации.

### Проверка

In [ ]:
spark.sql("""
    SELECT 
        category_name,
        COUNT(*) AS item_count,
        AVG(price) AS avg_price,
        MAX(price) AS max_price
    FROM ebay_listings_optimized
    WHERE price > 0
    GROUP BY category_name
    HAVING category_name = 'Electronics' -- <-----
    ORDER BY category_name
""").show()

+-------------+----------+------------------+---------+
|category_name|item_count|         avg_price|max_price|
+-------------+----------+------------------+---------+
|  Electronics|       315|130.29438095238095|  1248.46|
+-------------+----------+------------------+---------+



In [ ]:
spark.sql("""
select avg(price), count(*), max(price) from 
(SELECT price FROM ebay_listings_optimized WHERE category_name = 'Electronics' AND price > 0);
""").show()


+------------------+--------+----------+
|        avg(price)|count(1)|max(price)|
+------------------+--------+----------+
|130.29438095238095|     315|   1248.46|
+------------------+--------+----------+



### 2

Выявить связь между статусом продавца и условиями доставки. Вывести топ-10 стран (location_country), в которых средний процент положительных отзывов (seller_feedback_percentage) у «топ-продавцов» (top_rated_buying_experience = true) выше всего

---

In [7]:
spark.sql("""
    SELECT 
        location_country,
        AVG(seller_feedback_percentage) AS avg_positive_feedback
    FROM ebay_listings_optimized
    WHERE top_rated_buying_experience = true
      AND seller_feedback_percentage IS NOT NULL
    GROUP BY location_country
    ORDER BY avg_positive_feedback DESC
    LIMIT 10
""").show()

+----------------+---------------------+
|location_country|avg_positive_feedback|
+----------------+---------------------+
|              ME|                100.0|
|              MO|                100.0|
|              MK|                100.0|
|              MW|                100.0|
|              NA|                100.0|
|              MU|                100.0|
|              SN|                100.0|
|              KG|                100.0|
|              UY|                100.0|
|              CI|                100.0|
+----------------+---------------------+



### Пояснение

- **Что считается:** для каждой страны средний процент положительных отзывов (`seller_feedback_percentage`) только у продавцов с `top_rated_buying_experience = true`.  
- **Почему формула корректна:** среднее арифметическое корректно отражает общий уровень доверия к топ-продавцам в стране. Игнорируются NULL-значения (AVG автоматически их пропускает).  
- **Проверка:** Взять одну страну, например, `'US'`, и вручную посчитать средний процент:  
  ```sql
  SELECT seller_feedback_percentage FROM ebay_listings_optimized 
  WHERE location_country = 'US' AND top_rated_buying_experience = true AND seller_feedback_percentage IS NOT NULL;
  ```  
  Сравнить с результатом агрегации.

### Проверка


In [20]:

spark.sql("""
          select avg(seller_feedback_percentage) from (
SELECT seller_feedback_percentage FROM ebay_listings_optimized 
  WHERE location_country = 'US' AND top_rated_buying_experience = true AND seller_feedback_percentage IS NOT NULL
          );
""").show()


+-------------------------------+
|avg(seller_feedback_percentage)|
+-------------------------------+
|               97.6367813013053|
+-------------------------------+



средний процент ниже чем у последней страны в топе - us не попадает

### 3

Создать новый столбец total_price_usd, который суммирует price и shipping_cost. Вывести список товаров, где эта сумма превышает 500 USD, а доставка является фиксированной (shipping_cost_type = 'FIXED')

---

In [8]:
spark.sql("""
    SELECT 
        itemid,
        title,
        price,
        shipping_cost,
        (price + shipping_cost) AS total_price_usd,
        shipping_cost_type
    FROM ebay_listings_optimized
    WHERE (price + shipping_cost) > 500
      AND shipping_cost_type = 'FIXED'
    ORDER BY total_price_usd DESC
""").show()

+------------+---------------------+---------+-------------+-----------------+------------------+
|      itemid|                title|    price|shipping_cost|  total_price_usd|shipping_cost_type|
+------------+---------------------+---------+-------------+-----------------+------------------+
|227332977407| Lego Pokemon 7215...|999999.99|         75.0|       1000074.99|             FIXED|
|168308219349| bricks porsche rs...|1000000.0|         15.0|        1000015.0|             FIXED|
|236675213386| Orange & Red Sush...|999999.99|         10.0|       1000009.99|             FIXED|
|157780805826| Pokemon Base Set ...|1000000.0|         6.99|       1000006.99|             FIXED|
|306894108935| 2002 POKEMON LEGE...|1000000.0|         5.99|       1000005.99|             FIXED|
|406831816952| Ty 4005 Beanie Ba...|1000000.0|          0.0|        1000000.0|             FIXED|
|406825938053| Ty 4005 Beanie Ba...|1000000.0|          0.0|        1000000.0|             FIXED|
|406831773685| Ty Be

### Пояснение

- **Что считается:** `total_price_usd = price + shipping_cost`. Выводятся товары, где сумма > 500 и `shipping_cost_type = 'FIXED'`.  
- **Почему формула корректна:** предполагается, что обе цены в одинаковой валюте (или уже сконвертированы в USD). Суммирование даёт полную стоимость для покупателя.  
- **Проверка:** Для одного товара проверить арифметику:  
  ```sql
  SELECT price, shipping_cost, price+shipping_cost FROM ebay_listings_optimized WHERE itemid = '<id>';
  ```  
  Убедиться, что условие `>500` применяется правильно.

### Проверка


In [22]:

spark.sql("""
SELECT price, shipping_cost, price+shipping_cost FROM ebay_listings_optimized WHERE itemid = '306894108935';
""").show()


+---------+-------------+-----------------------+
|    price|shipping_cost|(price + shipping_cost)|
+---------+-------------+-----------------------+
|1000000.0|         5.99|             1000005.99|
+---------+-------------+-----------------------+



арифметика совпадает

# 4

В каждой подкатегории (sub_category_name) найти по одному продавцу с самым высоким показателем seller_feedback_score

---

In [11]:
spark.sql("""
    WITH ranked AS (
        SELECT 
            sub_category_name, 
            seller_name, 
            seller_feedback_score,
            ROW_NUMBER() OVER (
                PARTITION BY sub_category_name 
                ORDER BY seller_feedback_score DESC, seller_name
            ) AS rn
        FROM 
            ebay_listings_optimized 
        WHERE seller_feedback_score IS NOT NULL
    )
    SELECT 
        sub_category_name, 
        seller_name, 
        seller_feedback_score 
    FROM ranked 
    WHERE rn = 1 
    ORDER BY sub_category_name
""").show()

+-----------------+------------------+---------------------+
|sub_category_name|       seller_name|seller_feedback_score|
+-----------------+------------------+---------------------+
|             NULL|          the_nile|              1274042|
|$20, Double Eagle|       fb64-4glrx-|                    0|
|                1|       jLuLeCbjRSa|              5680850|
|            10033|       CV4X1GnWR6O|               696685|
|            10063|   zs_zweiradteile|               880616|
|           101925|        wuxianshen|                    9|
|            10285|         mohfaisal|                    6|
|           103430|        city-green|               689963|
|           103431|       yj3mEOG2RoO|                  542|
|             1039|arcadia_modellismo|                17181|
|           105559|       zoraoptwtpc|                32148|
|             1059|       iWiCELLQSL2|              1205604|
|           106256|           xyplus6|                 3556|
|           106981|     

### Пояснение

- **Что считается:** для каждой подкатегории выбирается продавец с наибольшим `seller_feedback_score`. При равных – берётся первый по имени.  
- **Почему формула корректна:** `ROW_NUMBER` гарантирует одного продавца на подкатегорию. Если важны все продавцы с максимальным баллом, можно использовать `RANK()`. Задание требует “по одному”, поэтому наш вариант соответствует.  
- **Проверка:** Для выбранной подкатегории выполнить:  
  ```sql
  SELECT MAX(seller_feedback_score) FROM ebay_listings_optimized WHERE sub_category_name = '<name>';
  ```  
  Затем найти продавцов с этим баллом и убедиться, что выбранный совпадает с одним из них.

### Проверка


In [24]:

spark.sql("""
SELECT MAX(seller_feedback_score) FROM ebay_listings_optimized WHERE sub_category_name = '103430';
""").show()


+--------------------------+
|max(seller_feedback_score)|
+--------------------------+
|                    689963|
+--------------------------+



score совпадает

# 5

Посчитать количество пропущенных значений (NULL) в колонке estimated_delivery_days для каждой даты выгрузки (snapshot_dt). Вывести только те даты, где количество пропусков больше 10% от общего числа записей за этот день.

---

In [12]:
spark.sql("""
    SELECT 
        snapshot_dt, 
        COUNT(*) AS total_rows,
        SUM(CASE 
                WHEN estimated_delivery_days IS NULL THEN 1 ELSE 0 END
        ) AS null_count,
        ROUND(
            SUM(CASE 
                    WHEN estimated_delivery_days IS NULL THEN 1 ELSE 0 END
            )*100.0/COUNT(*),2
        ) AS null_percentage
    FROM ebay_listings_optimized 
    WHERE snapshot_dt IS NOT NULL
    GROUP BY snapshot_dt
    HAVING (
        SUM(CASE 
            WHEN estimated_delivery_days IS NULL THEN 1 ELSE 0 END
        )*100.0/COUNT(*)
    ) > 10
    ORDER BY snapshot_dt
""").show()

+-----------+----------+----------+---------------+
|snapshot_dt|total_rows|null_count|null_percentage|
+-----------+----------+----------+---------------+
| 2026-02-28|     26223|      7689|          29.32|
| 2026-03-01|     61907|     22671|          36.62|
| 2026-03-02|       127|        29|          22.83|
| 2026-03-03|    102470|     22743|          22.19|
| 2026-03-04|    145406|     36247|          24.93|
| 2026-03-05|    151337|     39292|          25.96|
| 2026-03-06|    151683|     34208|          22.55|
| 2026-03-07|    138809|     34068|          24.54|
| 2026-03-08|    137803|     37026|          26.87|
| 2026-03-09|    112467|     27159|          24.15|
| 2026-03-10|    118220|     27307|          23.10|
| 2026-03-15|    137824|     39891|          28.94|
| 2026-03-16|    132314|     33485|          25.31|
| 2026-03-17|    150102|     36265|          24.16|
| 2026-03-18|    140551|     35543|          25.29|
| 2026-03-19|    136883|     33343|          24.36|
| 2026-03-20

### Пояснение

- **Что считается:** для каждой даты – процент записей, где `estimated_delivery_days` равно NULL.  
- **Почему формула корректна:** `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` точно подсчитывает NULL, а общее число записей – через `COUNT(*)`. Доля вычисляется как `(null_count * 100.0 / total_rows)`.  
- **Проверка:** Взять одну дату, например `'2026-03-01'`, и вручную посчитать:  
  ```sql
  SELECT 
      COUNT(*) AS total,
      SUM(CASE WHEN estimated_delivery_days IS NULL THEN 1 ELSE 0 END) AS nulls
  FROM ebay_listings_optimized 
  WHERE snapshot_dt = '2026-03-01';
  ```  
  Проверить, что процент совпадает с выводом основного запроса.

### Проверка


In [25]:

spark.sql("""
SELECT 
      COUNT(*) AS total,
      SUM(CASE WHEN estimated_delivery_days IS NULL THEN 1 ELSE 0 END) AS nulls
  FROM ebay_listings_optimized 
  WHERE snapshot_dt = '2026-03-01';
""").show()


+-----+-----+
|total|nulls|
+-----+-----+
|61907|22671|
+-----+-----+



результат совпадает